# VectorStore

## Carregamento de bibliotecas

In [20]:
from  langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore, FAISS
import faiss
import os
import warnings
from dotenv import load_dotenv
import httpx

warnings.filterwarnings("ignore")
http_client = httpx.Client(verify=False)

In [3]:
os.chdir(r'c:\Users\francisco.bneto\Documents\gen-ai-formation')
print(os.getcwd())

c:\Users\francisco.bneto\Documents\gen-ai-formation


In [4]:
load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")
print("Chave carregada:", api_key[:10] + "...")

Chave carregada: sk-or-v1-c...


## Carregamento dos documentos e criação de chunks

In [8]:
documentos = DirectoryLoader("./documentos", glob="*.pdf").load()

In [11]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = splitter.split_documents(documentos)

In [12]:
len(chunks)

204

In [14]:
chunks[10].page_content

'programas de recompensas relacionados ao seu cartão;\n\nb) Tenha uma Garantia mínima do Fornecedor (fabricante e/ou marca da loja) de 3 (três) meses; c) Não tenham um período de Garantia do Fornecedor maior que 3 (três) anos; d) A compra coberta precisa ter uma Garantia válida, ou seja: (1) Deve existir disponibilidade de uma rede autorizada do Fornecedor para consertos e peças no País de Residência do Portador de Cartão; (2) confirmação de que o produto qualifica-se para a Garantia no País de Residência do Portador de Cartão; (3) a Garantia contenha tudo que é coberto ou não; (4) o período de cobertura; (5) o que o Fornecedor terá de fazer para solucionar o problema; e (6) com quem entrar em contato para obter os serviços oferecidos na Garantia.'

## Carregamento do modelo de embeddings

In [6]:
hf_emb_model = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-small")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4233.92it/s]


## Indexar documentos no banco de dados vetorial

### InMemoryVectorStore

In [ ]:
vectorstore = InMemoryVectorStore.from_documents(
    documents=chunks, embedding=hf_emb_model
)

In [17]:
retriever = vectorstore.as_retriever(
    search_kwargs={'k': 3}
)

retriever.invoke('Seguro viagem')

[Document(id='fca50896-0239-4ccc-b960-1218315adc13', metadata={'source': 'documentos\\GTB_platinum_Nov23.pdf'}, page_content='Esta cobertura fornece um benefício máximo de até USD† 25.000 por Pessoa Elegível.\n\nO Seguro Viagem cobre despesas médicas relacionadas a lesões ou doenças súbitas que ocorram durante uma Viagem Segurada Internacional fora do Brasil, inclusive cobertura para COVID-19, SARS-Cov-2 e qualquer mutação ou variação do SARS-CoV-2, sujeito a todos os termos e condições aplicáveis da apólice. A cobertura se aplica para todas as Viagens Seguradas Internacionais iniciadas a partir de 1º de maio de 2021, independentemente de quando a viagem foi comprada ou destino internacional da Viagem Segurada, exceto Brasil. No entanto, não será aplicável se a lesão ou doença ocorrer antes do início da viagem.\n\nVersão: novembro/2021'),
 Document(id='62a446c7-83f5-4e91-a5d1-1e3c74650544', metadata={'source': 'documentos\\GTB_platinum_Nov23.pdf'}, page_content='feita se não é superior

### FAISS

In [23]:
vectorstore_faiss = FAISS.from_documents(
    documents=chunks, embedding=hf_emb_model
)

retriever = vectorstore_faiss.as_retriever(search_kwargs={'k': 3})

retriever.invoke("Sala VIP")

[Document(id='ae0119a9-d6a7-4c89-a2a9-9d33fbf58ca8', metadata={'source': 'documentos\\GTB_gold_Nov23.pdf'}, page_content='5) Equipamentos médicos, fisioterapêuticos, ortodônticos ou relacionados à área de saúde em geral. Salvo em caso de definição contrária;\n\n6) Extintores de incêndio, espelhos e vidros em geral, lâmpadas, geradores de energia, painéis solares, letreiros elétricos, lentes, óculos, telescópios, microscópio, carregadores;\n\n7) Cheques de viagem; bilhetes de algum tipo; instrumentos negociáveis; ouro ou prata em barras; dinheiro ou equivalentes; moedas raras ou preciosas; propriedade filatélica ou numismática;\n\n8) Plantas, projetos, manuscritos, modelos, debuxos e moldes, livros de contabilidade, certidões,\n\nregistros e documentos de qualquer espécie;\n\nVersão: novembro 2023 2021\n\n6\n\n9) Bebidas, comestíveis, perfumes, cosméticos, remédios e semelhantes;\n\n10) Veículos motorizados, embarcações, barcos a motor, aviões, motocicletas e similares, bem como suas pe